In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_4.pickle

In [4]:
%%RecordEvent
%%time
### cell 4 ###

def create_dataframe_of_counts_16(dataframe, column, rename_index, rename_column, return_percentages=False):
    # Drop the first row to match the original slicing
    dataframe = dataframe.drop(index=[dataframe.index[0]])
    # Compute counts directly on GPU without explicit column indexing
    df_counts = dataframe.value_counts(subset=[column]).reset_index()
    # Rename columns
    df_counts = df_counts.rename({column: rename_index, 'count': rename_column}, axis='columns')
    if return_percentages:
        total = df_counts[rename_column].sum()
        df_counts = df_counts.assign(**{rename_column: df_counts[rename_column] * 100 / total})
    return df_counts

percentages_per_country_df = create_dataframe_of_counts_16(
    responses_df_2022_cell10,
    'In which country do you currently reside?',
    'country',
    '% of respondents',
    return_percentages=False
)
percentages_per_country_df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   country           58 non-null     object
 1   % of respondents  58 non-null     int64
dtypes: int64(1), object(1)
memory usage: 1.2+ KB
CPU times: user 170 ms, sys: 15.6 ms, total: 186 ms
Wall time: 193 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_4_try_2.pickle